<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/2.0%20Feature_%20Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The primary purpose of this notebook is to control what features are fed in a particular notebook\model.
To ensure convenient feature_engineering from transaction items, particularly for machine learning training and production, python functions were created and stored in an AWS S3 bucket. This is to esnure that all notebooks beyond this one, only pull data from the database through a consistent channel, using the `v_clean_sales_analytics` via the functions tailored each notebook's needs. That said, this notebooks purpose is the creation of a `feature_engineering` function that will deliver features for revenue, and customer churn, prediction, as well a `customer_segmentation_features` which is depended on data produced by the features engineering, to serve the customer segmentation training and assignments.

In [25]:
#!pip install boto3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata

from sqlalchemy import create_engine, text, inspect

import boto3
import sys
import os
import importlib

bucket_name = 'sales-data-analytics-portfolio-2026'

#Intializing s3 client with credentials
s3 = boto3.client(
    "s3",
    aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name = 'eu-north-1'
)

with open("feature_engineering.py", "w") as f:
    f.write("""
import pandas as pd
import numpy as np
def feature_engineering(df):

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    end_date = df["InvoiceDate"].max()

    cust_data = df.groupby("CustomerID")[["InvoiceDate", "StockCode", "Quantity", "UnitPrice", "Revenue"]].agg(
       FirstPurchase = ("InvoiceDate", "min"),
       LastPurchase = ("InvoiceDate", "max"),
       Frequency = ("InvoiceDate", "nunique"),
       ProductDiversity = ("StockCode", "nunique"),
       AvgUnitPrice = ("UnitPrice", "mean"),
       AvgQuantity = ("Quantity", "mean"),
       AOV = ("Revenue", "mean"),
       TotalRevenue = ("Revenue", "sum")
    )

    cust_data["Recency"] = 1 + (end_date - cust_data["LastPurchase"]).dt.days
    cust_data["Tenure"] = 1 + (end_date - cust_data["FirstPurchase"]).dt.days
    cust_data["ObservedLifeSpan"] = 1 + (cust_data["LastPurchase"] - cust_data["FirstPurchase"]).dt.days

    cust_data.drop(["FirstPurchase","LastPurchase"], axis = 1, inplace = True)

    cust_data["RecencyToTenure"] = cust_data["Recency"]/(cust_data["Tenure"])
    cust_data["ActivePurchaseDensity"] = cust_data["Frequency"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimePurchaseDensity"] = cust_data["Frequency"]/cust_data["Tenure"]

    cust_data["ActiveMRate"] = cust_data["TotalRevenue"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimeMRate"] = cust_data["TotalRevenue"]/cust_data["Tenure"]

    ipi = (df[["CustomerID", "InvoiceDate"]].sort_values(["CustomerID", "InvoiceDate"], ascending = False))
    ipi = ipi.groupby(["CustomerID", "InvoiceDate"]).size().reset_index()
    ipi["InterPurchaseInterval"] = np.where(ipi["CustomerID"].shift(1) == ipi["CustomerID"], (ipi["InvoiceDate"] - ipi["InvoiceDate"].shift(1)).dt.days + 1, 1)
    ipi = ipi.groupby("CustomerID").agg(
        AvgIPI = ("InterPurchaseInterval", "mean")
    ).reset_index()

    cust_data = cust_data.merge(ipi[["CustomerID", "AvgIPI"]], on = "CustomerID", how = "left")

    cust_data["RelativeSilence"] = cust_data["Recency"]/cust_data["AvgIPI"]

    return cust_data"""
)


# Uploading file to s3 bucket
s3.upload_file('feature_engineering.py', bucket_name, 'functions/feature_engineering.py')
print(f'Successfully uploaded file to {bucket_name} bucket')

s3.download_file(bucket_name, 'functions/feature_engineering.py', 'feature_engineering.py')

sys.path.append(os.getcwd())

import feature_engineering

importlib.reload(feature_engineering)

from feature_engineering import feature_engineering

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

query = """
SELECT *
FROM v_clean_sales_analytics
WHERE "InvoiceDate" <= '2011-08-31'
"""

df = pd.read_sql(text(query), con = engine)

df = feature_engineering(df)
print('Preview of Features')
print(df.info())
df

Successfully uploaded file to sales-data-analytics-portfolio-2026 bucket
Preview of Features
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3317 entries, 0 to 3316
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CustomerID               3317 non-null   float64
 1   Frequency                3317 non-null   int64  
 2   ProductDiversity         3317 non-null   int64  
 3   AvgUnitPrice             3317 non-null   float64
 4   AvgQuantity              3317 non-null   float64
 5   AOV                      3317 non-null   float64
 6   TotalRevenue             3317 non-null   float64
 7   Recency                  3317 non-null   int64  
 8   Tenure                   3317 non-null   int64  
 9   ObservedLifeSpan         3317 non-null   int64  
 10  RecencyToTenure          3317 non-null   float64
 11  ActivePurchaseDensity    3317 non-null   float64
 12  LifetimePurchaseDensity  3317 non-null 

,CustomerID,Frequency,ProductDiversity,AvgUnitPrice,AvgQuantity,AOV,TotalRevenue,Recency,Tenure,ObservedLifeSpan,RecencyToTenure,ActivePurchaseDensity,LifetimePurchaseDensity,ActiveMRate,LifetimeMRate,AvgIPI,RelativeSilence
0,12346.0,1,1,1.040000,74215.000000,77183.600000,77183.60,226,226,1,1.000000,1.000000,0.004425,77183.600000,341.520354,1.000000,226.000000
1,12347.0,5,82,2.797661,12.822581,22.506935,2790.86,30,268,239,0.111940,0.020921,0.018657,11.677238,10.413657,48.600000,0.617284
2,12348.0,3,22,4.864643,75.857143,53.115714,1487.24,149,259,111,0.575290,0.027027,0.011583,13.398559,5.742239,37.666667,3.955752
3,12350.0,1,17,3.841176,11.588235,19.670588,334.40,211,211,1,1.000000,1.000000,0.004739,334.400000,1.584834,1.000000,211.000000
4,12352.0,4,26,27.449474,6.684211,41.100263,1561.81,163,197,35,0.827411,0.114286,0.020305,44.623143,7.927970,9.500000,17.157895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3312,18280.0,1,10,4.765000,4.500000,18.060000,180.60,178,178,1,1.000000,1.000000,0.005618,180.600000,1.014607,1.000000,178.000000
3313,18281.0,1,7,5.622857,7.714286,11.545714,80.82,81,81,1,1.000000,1.000000,0.012346,80.820000,0.997778,1.000000,81.000000
3314,18282.0,1,7,5.552857,10.714286,14.315714,100.21,27,27,1,1.000000,1.000000,0.037037,100.210000,3.711481,1.000000,27.000000
3315,18283.0,8,183,1.733634,1.655172,2.525013,951.93,49,238,190,0.205882,0.042105,0.033613,5.010158,3.999706,24.625000,1.989848


In [26]:
# creating function to pull features specifically for customer Segmentation from dataframe after passing feature_engineering function

import sys
import os

with open("segmentation_features.py", "w") as f:
    f.write("""def segmentation_features(df):
    features = [
    "CustomerID",
    "Tenure",
    "ObservedLifeSpan",
    "TotalRevenue",
    "Recency",
    "ProductDiversity",
    "Frequency"]

    return df[features]""")

s3.upload_file("segmentation_features.py", bucket_name, "functions/segmentation_features.py")
print("Successfully uploaded Segmentation function to S3 Bucket")

s3.download_file(bucket_name, "functions/segmentation_features.py", "segmentation_features.py")
print("\nSuccessfully download Segmentation function from AWA S3")

sys.path.append(os.getcwd())
import segmentation_features

importlib.reload(segmentation_features)

from segmentation_features import segmentation_features

df = segmentation_features(df)
print("\nPreview of features particular to Customer Segmentation")
df.head()

Successfully uploaded Segmentation function to S3 Bucket

Successfully download Segmentation function from AWA S3

Preview of features particular to Customer Segmentation


,CustomerID,Tenure,ObservedLifeSpan,TotalRevenue,Recency,ProductDiversity,Frequency
0,12346.0,226,1,77183.60,226,1,1
1,12347.0,268,239,2790.86,30,82,5
2,12348.0,259,111,1487.24,149,22,3
3,12350.0,211,1,334.40,211,17,1
4,12352.0,197,35,1561.81,163,26,4
